In [6]:
# ============================================================
# STEP 7 — CLAIMS DATA PROFILING
# ============================================================

import pandas as pd

# Load raw Claims CSV
claims_raw = pd.read_csv("MedCore_RCM_Data/Claims.csv")

print("========== CLAIMS DATA PROFILING ==========")

# ------------------------------------------------------------
# 1. Shape
# ------------------------------------------------------------

print("\n1. Dataset Shape:")
print(f"Rows    : {claims_raw.shape[0]:,}")
print(f"Columns : {claims_raw.shape[1]}")


# ------------------------------------------------------------
# 2. Column names and data types
# ------------------------------------------------------------

print("\n2. Data Types:")
print(claims_raw.dtypes)


# ------------------------------------------------------------
# 3. Missing values
# ------------------------------------------------------------

print("\n3. Missing Values:")

missing_values = (
    claims_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(claims_raw)
    * 100
).round(2)

display(missing_values)


# ------------------------------------------------------------
# 4. Duplicate Claim IDs
# ------------------------------------------------------------

print("\n4. Duplicate Claim IDs:")
print(
    claims_raw["Claim_ID"].duplicated().sum()
)


# ------------------------------------------------------------
# 5. Unique values
# ------------------------------------------------------------

print("\n5. Unique Values:")

for column in [
    "Specialty",
    "Procedure_Code",
    "Procedure_Name",
    "Denial_Flag",
    "Denial_Reason",
    "Claim_Status"
]:
    print(f"\n{column}:")
    print(claims_raw[column].nunique())


# ------------------------------------------------------------
# 6. Claim amount checks
# ------------------------------------------------------------

print("\n6. Claim Amount Summary:")
print(
    claims_raw["Claim_Amount"].describe()
)


print("\nNegative Claim Amounts:")
print(
    (claims_raw["Claim_Amount"] < 0).sum()
)

print("\nZero Claim Amounts:")
print(
    (claims_raw["Claim_Amount"] == 0).sum()
)


# ------------------------------------------------------------
# 7. Date validation
# ------------------------------------------------------------

claims_raw["Service_Date"] = pd.to_datetime(
    claims_raw["Service_Date"],
    errors="coerce"
)

print("\n7. Service Date Range:")
print("Minimum:", claims_raw["Service_Date"].min())
print("Maximum:", claims_raw["Service_Date"].max())

print("\nInvalid Service Dates:")
print(
    claims_raw["Service_Date"].isna().sum()
)


# ------------------------------------------------------------
# 8. Denial flag distribution
# ------------------------------------------------------------

print("\n8. Denial Flag Distribution:")
display(
    claims_raw["Denial_Flag"]
    .value_counts()
    .to_frame("Claim_Count")
)


# ------------------------------------------------------------
# 9. Claim status distribution
# ------------------------------------------------------------

print("\n9. Claim Status Distribution:")
display(
    claims_raw["Claim_Status"]
    .value_counts()
    .to_frame("Claim_Count")
)


# ------------------------------------------------------------
# 10. Denial reason distribution
# ------------------------------------------------------------

print("\n10. Denial Reason Distribution:")
display(
    claims_raw["Denial_Reason"]
    .value_counts(dropna=False)
    .to_frame("Claim_Count")
)

========== CLAIMS DATA PROFILING ==========

1. Dataset Shape:
Rows    : 50,000
Columns : 12

2. Data Types:
Claim_ID           object
Patient_ID         object
Provider_ID        object
Payer_ID           object
Service_Date       object
Specialty          object
Procedure_Code     object
Procedure_Name     object
Claim_Amount      float64
Denial_Flag         int64
Denial_Reason      object
Claim_Status       object
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Claim_ID,0,0.00
Patient_ID,0,0.00
Provider_ID,0,0.00
Payer_ID,0,0.00
Service_Date,0,0.00
Specialty,0,0.00
Procedure_Code,0,0.00
Procedure_Name,0,0.00
Claim_Amount,0,0.00
Denial_Flag,0,0.00



4. Duplicate Claim IDs:
0

5. Unique Values:

Specialty:
8

Procedure_Code:
24

Procedure_Name:
24

Denial_Flag:
2

Denial_Reason:
7

Claim_Status:
3

6. Claim Amount Summary:
count     50000.000000
mean      15417.896115
std       40691.284905
min         500.100000
25%        1651.865000
50%        3464.650000
75%        8251.737500
max      349962.410000
Name: Claim_Amount, dtype: float64

Negative Claim Amounts:
0

Zero Claim Amounts:
0

7. Service Date Range:
Minimum: 2025-01-01 00:00:00
Maximum: 2025-12-31 00:00:00

Invalid Service Dates:
0

8. Denial Flag Distribution:


,Claim_Count
Denial_Flag,
0,45958
1,4042



9. Claim Status Distribution:


,Claim_Count
Claim_Status,
Paid,43224
Denied,4042
Pending,2734



10. Denial Reason Distribution:


,Claim_Count
Denial_Reason,
NaN,45958
Documentation Missing,849
Coding Error,836
Medical Necessity,692
Eligibility Issue,637
Missing Authorization,566
Timely Filing,280
Duplicate Claim,182


In [4]:
# ============================================================
# CLAIMS — BUSINESS RULE VALIDATION
# ============================================================

print("========== CLAIM BUSINESS RULE VALIDATION ==========")

# Rule 1:
# Every denied claim should have Denial_Flag = 1

rule_1 = (
    (claims_raw["Claim_Status"] == "Denied") &
    (claims_raw["Denial_Flag"] != 1)
).sum()

print("\nDenied status but Denial_Flag != 1:")
print(rule_1)


# Rule 2:
# Every Denial_Flag = 1 claim should have Denial_Reason

rule_2 = (
    (claims_raw["Denial_Flag"] == 1) &
    (claims_raw["Denial_Reason"].isna())
).sum()

print("\nDenied claims with missing Denial_Reason:")
print(rule_2)


# Rule 3:
# Non-denied claims should not have a Denial_Reason

rule_3 = (
    (claims_raw["Denial_Flag"] == 0) &
    (claims_raw["Denial_Reason"].notna())
).sum()

print("\nNon-denied claims with a Denial_Reason:")
print(rule_3)


# Rule 4:
# Claim_Status should only contain expected values

expected_statuses = {
    "Paid",
    "Denied",
    "Pending"
}

unexpected_statuses = (
    ~claims_raw["Claim_Status"].isin(expected_statuses)
).sum()

print("\nUnexpected Claim_Status values:")
print(unexpected_statuses)

========== CLAIM BUSINESS RULE VALIDATION ==========

Denied status but Denial_Flag != 1:
0

Denied claims with missing Denial_Reason:
0

Non-denied claims with a Denial_Reason:
0

Unexpected Claim_Status values:
0


In [7]:
import pandas as pd
# ============================================================
# STEP 8 — AUTHORIZATION DATA PROFILING
# ============================================================

authorization_raw = pd.read_csv(
    "MedCore_RCM_Data/Authorization.csv"
)

print("========== AUTHORIZATION DATA PROFILING ==========")

# 1. Dataset shape
print("\n1. Dataset Shape:")
print(f"Rows    : {authorization_raw.shape[0]:,}")
print(f"Columns : {authorization_raw.shape[1]}")


# 2. Data types
print("\n2. Data Types:")
print(authorization_raw.dtypes)


# 3. Missing values
print("\n3. Missing Values:")

missing_values = (
    authorization_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(authorization_raw)
    * 100
).round(2)

display(missing_values)


# 4. Duplicate Claim IDs
print("\n4. Duplicate Claim IDs:")
print(
    authorization_raw["Claim_ID"].duplicated().sum()
)


# 5. Unique values
print("\n5. Unique Values:")

for column in [
    "Authorization_Required",
    "Authorization_Status"
]:
    print(f"\n{column}:")
    print(authorization_raw[column].value_counts(dropna=False))


# 6. Authorization ID checks
print("\n6. Authorization ID Checks:")

print(
    "Missing Authorization IDs:",
    authorization_raw["Authorization_ID"].isna().sum()
)

print(
    "Unique Authorization IDs:",
    authorization_raw["Authorization_ID"].nunique()
)


# 7. Date conversion
authorization_raw["Request_Date"] = pd.to_datetime(
    authorization_raw["Request_Date"],
    errors="coerce"
)

authorization_raw["Approval_Date"] = pd.to_datetime(
    authorization_raw["Approval_Date"],
    errors="coerce"
)

print("\n7. Date Range:")

print(
    "Request Date:",
    authorization_raw["Request_Date"].min(),
    "to",
    authorization_raw["Request_Date"].max()
)

print(
    "Approval Date:",
    authorization_raw["Approval_Date"].min(),
    "to",
    authorization_raw["Approval_Date"].max()
)


# 8. Invalid dates
print("\n8. Invalid Dates:")

print(
    "Invalid Request Dates:",
    authorization_raw["Request_Date"].isna().sum()
)

print(
    "Invalid Approval Dates:",
    authorization_raw["Approval_Date"].isna().sum()
)


# 9. First 10 records
print("\n9. First 10 Records:")
display(authorization_raw.head(10))

========== AUTHORIZATION DATA PROFILING ==========

1. Dataset Shape:
Rows    : 50,000
Columns : 6

2. Data Types:
Claim_ID                  object
Authorization_Required      bool
Authorization_Status      object
Request_Date              object
Approval_Date             object
Authorization_ID          object
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Claim_ID,0,0.00
Authorization_Required,0,0.00
Authorization_Status,0,0.00
Request_Date,35309,70.62
Approval_Date,37337,74.67
Authorization_ID,35309,70.62



4. Duplicate Claim IDs:
0

5. Unique Values:

Authorization_Required:
Authorization_Required
False    33324
True     16676
Name: count, dtype: int64

Authorization_Status:
Authorization_Status
Not Required     33324
Approved         12663
Not Requested     1985
Pending           1164
Denied             864
Name: count, dtype: int64

6. Authorization ID Checks:
Missing Authorization IDs: 35309
Unique Authorization IDs: 14691

7. Date Range:
Request Date: 2024-12-18 00:00:00 to 2026-01-04 00:00:00
Approval Date: 2024-12-19 00:00:00 to 2026-01-07 00:00:00

8. Invalid Dates:
Invalid Request Dates: 35309
Invalid Approval Dates: 37337

9. First 10 Records:


,Claim_ID,Authorization_Required,Authorization_Status,Request_Date,Approval_Date,Authorization_ID
0,CL000001,False,Not Required,NaT,NaT,NaN
1,CL000002,True,Approved,2025-09-27,2025-09-28,AUTH000002
2,CL000003,False,Not Required,NaT,NaT,NaN
3,CL000004,False,Not Required,NaT,NaT,NaN
4,CL000005,False,Not Required,NaT,NaT,NaN
5,CL000006,False,Not Required,NaT,NaT,NaN
6,CL000007,False,Not Required,NaT,NaT,NaN
7,CL000008,False,Not Required,NaT,NaT,NaN
8,CL000009,False,Not Required,NaT,NaT,NaN
9,CL000010,False,Not Required,NaT,NaT,NaN


In [3]:
# ============================================================
# AUTHORIZATION — BUSINESS RULE VALIDATION
# ============================================================

print("========== AUTHORIZATION BUSINESS RULE VALIDATION ==========")

# Rule 1:
# Approved authorization should have a request date
rule_1 = (
    (authorization_raw["Authorization_Status"] == "Approved") &
    (authorization_raw["Request_Date"].isna())
).sum()

print("\nApproved but Request_Date is missing:")
print(rule_1)


# Rule 2:
# Approved authorization should have an approval date
rule_2 = (
    (authorization_raw["Authorization_Status"] == "Approved") &
    (authorization_raw["Approval_Date"].isna())
).sum()

print("\nApproved but Approval_Date is missing:")
print(rule_2)


# Rule 3:
# Approved authorization should have an Authorization ID
rule_3 = (
    (authorization_raw["Authorization_Status"] == "Approved") &
    (authorization_raw["Authorization_ID"].isna())
).sum()

print("\nApproved but Authorization_ID is missing:")
print(rule_3)


# Rule 4:
# Not Requested should not have a request date
rule_4 = (
    (authorization_raw["Authorization_Status"] == "Not Requested") &
    (authorization_raw["Request_Date"].notna())
).sum()

print("\nNot Requested but Request_Date exists:")
print(rule_4)


# Rule 5:
# Not Required should not have an Authorization ID
rule_5 = (
    (authorization_raw["Authorization_Status"] == "Not Required") &
    (authorization_raw["Authorization_ID"].notna())
).sum()

print("\nNot Required but Authorization_ID exists:")
print(rule_5)


# Rule 6:
# Approval should not happen before request
rule_6 = (
    authorization_raw["Approval_Date"].notna() &
    authorization_raw["Request_Date"].notna() &
    (authorization_raw["Approval_Date"] < authorization_raw["Request_Date"])
).sum()

print("\nApproval_Date before Request_Date:")
print(rule_6)

========== AUTHORIZATION BUSINESS RULE VALIDATION ==========

Approved but Request_Date is missing:
0

Approved but Approval_Date is missing:
0

Approved but Authorization_ID is missing:
0

Not Requested but Request_Date exists:
0

Not Required but Authorization_ID exists:
0

Approval_Date before Request_Date:
0


In [4]:
# ============================================================
# STEP 9 — PAYMENTS DATA PROFILING
# ============================================================

payments_raw = pd.read_csv(
    "MedCore_RCM_Data/Payments.csv"
)

print("========== PAYMENTS DATA PROFILING ==========")


# ------------------------------------------------------------
# 1. Dataset shape
# ------------------------------------------------------------

print("\n1. Dataset Shape:")
print(f"Rows    : {payments_raw.shape[0]:,}")
print(f"Columns : {payments_raw.shape[1]}")


# ------------------------------------------------------------
# 2. Data types
# ------------------------------------------------------------

print("\n2. Data Types:")
print(payments_raw.dtypes)


# ------------------------------------------------------------
# 3. Missing values
# ------------------------------------------------------------

print("\n3. Missing Values:")

missing_values = (
    payments_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(payments_raw)
    * 100
).round(2)

display(missing_values)


# ------------------------------------------------------------
# 4. Duplicate Claim IDs
# ------------------------------------------------------------

print("\n4. Duplicate Claim IDs:")
print(
    payments_raw["Claim_ID"].duplicated().sum()
)


# ------------------------------------------------------------
# 5. Recovery status distribution
# ------------------------------------------------------------

print("\n5. Recovery Status:")

display(
    payments_raw["Recovery_Status"]
    .value_counts(dropna=False)
    .to_frame("Claim_Count")
)


# ------------------------------------------------------------
# 6. Numeric amount summary
# ------------------------------------------------------------

print("\n6. Paid Amount Summary:")
print(
    payments_raw["Paid_Amount"].describe()
)

print("\nOutstanding Amount Summary:")
print(
    payments_raw["Outstanding_Amount"].describe()
)


# ------------------------------------------------------------
# 7. Negative amount checks
# ------------------------------------------------------------

print("\n7. Negative Amount Checks:")

print(
    "Negative Paid Amounts:",
    (payments_raw["Paid_Amount"] < 0).sum()
)

print(
    "Negative Outstanding Amounts:",
    (payments_raw["Outstanding_Amount"] < 0).sum()
)


# ------------------------------------------------------------
# 8. Zero payment checks
# ------------------------------------------------------------

print("\n8. Zero Paid Amounts:")

print(
    (payments_raw["Paid_Amount"] == 0).sum()
)


# ------------------------------------------------------------
# 9. Payment date conversion
# ------------------------------------------------------------

payments_raw["Payment_Date"] = pd.to_datetime(
    payments_raw["Payment_Date"],
    errors="coerce"
)

print("\n9. Payment Date Range:")

print(
    "Minimum:",
    payments_raw["Payment_Date"].min()
)

print(
    "Maximum:",
    payments_raw["Payment_Date"].max()
)


# ------------------------------------------------------------
# 10. Missing payment dates
# ------------------------------------------------------------

print("\n10. Missing Payment Dates:")

print(
    payments_raw["Payment_Date"].isna().sum()
)


# ------------------------------------------------------------
# 11. First 10 records
# ------------------------------------------------------------

print("\n11. First 10 Payment Records:")

display(
    payments_raw.head(10)
)

========== PAYMENTS DATA PROFILING ==========

1. Dataset Shape:
Rows    : 50,000
Columns : 5

2. Data Types:
Claim_ID               object
Paid_Amount           float64
Payment_Date           object
Recovery_Status        object
Outstanding_Amount    float64
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Claim_ID,0,0.00
Paid_Amount,0,0.00
Payment_Date,4337,8.67
Recovery_Status,0,0.00
Outstanding_Amount,0,0.00



4. Duplicate Claim IDs:
0

5. Recovery Status:


,Claim_Count
Recovery_Status,
Collected,43224
Pending,2734
Recovered,2439
Unrecovered,1603



6. Paid Amount Summary:
count     50000.000000
mean      12861.706549
std       36127.200524
min           0.000000
25%        1271.227500
50%        2780.530000
75%        5910.212500
max      344665.280000
Name: Paid_Amount, dtype: float64

Outstanding Amount Summary:
count     50000.000000
mean       2556.189567
std       14095.585823
min           0.000000
25%          83.767500
50%         220.365000
75%         838.895000
max      345316.590000
Name: Outstanding_Amount, dtype: float64

7. Negative Amount Checks:
Negative Paid Amounts: 0
Negative Outstanding Amounts: 0

8. Zero Paid Amounts:
4337

9. Payment Date Range:
Minimum: 2025-01-08 00:00:00
Maximum: 2026-03-01 00:00:00

10. Missing Payment Dates:
4337

11. First 10 Payment Records:


,Claim_ID,Paid_Amount,Payment_Date,Recovery_Status,Outstanding_Amount
0,CL000001,1005.48,2025-09-17,Collected,70.71
1,CL000002,7227.91,2025-10-25,Collected,206.74
2,CL000003,1135.71,2025-04-08,Collected,67.95
3,CL000004,1791.46,2026-02-03,Collected,76.26
4,CL000005,1569.67,2025-10-07,Collected,83.45
5,CL000006,4200.40,2025-10-29,Collected,267.91
6,CL000007,850.72,2025-08-12,Collected,58.92
7,CL000008,1382.49,2025-02-05,Collected,0.93
8,CL000009,4001.69,2025-12-29,Collected,135.65
9,CL000010,2173.23,2025-11-07,Collected,114.25


In [8]:
# ============================================================
# PAYMENTS — BUSINESS RULE VALIDATION
# ============================================================

print("========== PAYMENTS BUSINESS RULE VALIDATION ==========")

# ------------------------------------------------------------
# Bring Claim Amount from Claims table
# ------------------------------------------------------------

payment_check = payments_raw.merge(
    claims_raw[["Claim_ID", "Claim_Amount", "Service_Date"]],
    on="Claim_ID",
    how="left"
)

# ------------------------------------------------------------
# Rule 1: Every payment should have a matching claim amount
# ------------------------------------------------------------

rule_1 = payment_check["Claim_Amount"].isna().sum()

print("\nPayment records without matching Claim Amount:")
print(rule_1)


# ------------------------------------------------------------
# Rule 2: Paid + Outstanding should equal Claim Amount
# ------------------------------------------------------------

payment_check["Calculated_Total"] = (
    payment_check["Paid_Amount"]
    + payment_check["Outstanding_Amount"]
)

payment_check["Amount_Difference"] = (
    payment_check["Claim_Amount"]
    - payment_check["Calculated_Total"]
)

rule_2 = (
    payment_check["Amount_Difference"].abs() > 0.01
).sum()

print("\nClaims where Paid + Outstanding != Claim Amount:")
print(rule_2)


# ------------------------------------------------------------
# Rule 3: Zero Paid Amount should have no Payment Date
# ------------------------------------------------------------

rule_3 = (
    (payment_check["Paid_Amount"] == 0) &
    (payment_check["Payment_Date"].notna())
).sum()

print("\nZero Paid Amount but Payment Date exists:")
print(rule_3)


# ------------------------------------------------------------
# Rule 4: Positive Paid Amount should normally have Payment Date
# ------------------------------------------------------------

rule_4 = (
    (payment_check["Paid_Amount"] > 0) &
    (payment_check["Payment_Date"].isna())
).sum()

print("\nPositive Paid Amount but Payment Date is missing:")
print(rule_4)


# ------------------------------------------------------------
# Rule 5: Payment date should not be before service date
# ------------------------------------------------------------

rule_5 = (
    payment_check["Payment_Date"].notna() &
    payment_check["Service_Date"].notna() &
    (payment_check["Payment_Date"] < payment_check["Service_Date"])
).sum()

print("\nPayment Date before Service Date:")
print(rule_5)


# ------------------------------------------------------------
# Rule 6: Recovery status vs zero payment
# ------------------------------------------------------------

zero_paid_status = payment_check[
    payment_check["Paid_Amount"] == 0
]["Recovery_Status"].value_counts(dropna=False)

print("\nRecovery Status for Zero-Paid Claims:")
display(zero_paid_status.to_frame("Claim_Count"))


# ------------------------------------------------------------
# Rule 7: Recovery status vs positive payment
# ------------------------------------------------------------

positive_paid_status = payment_check[
    payment_check["Paid_Amount"] > 0
]["Recovery_Status"].value_counts(dropna=False)

print("\nRecovery Status for Positive-Paid Claims:")
display(positive_paid_status.to_frame("Claim_Count"))

========== PAYMENTS BUSINESS RULE VALIDATION ==========

Payment records without matching Claim Amount:
0

Claims where Paid + Outstanding != Claim Amount:
0

Zero Paid Amount but Payment Date exists:
0

Positive Paid Amount but Payment Date is missing:
0

Payment Date before Service Date:
0

Recovery Status for Zero-Paid Claims:


,Claim_Count
Recovery_Status,
Pending,2734
Unrecovered,1603



Recovery Status for Positive-Paid Claims:


,Claim_Count
Recovery_Status,
Collected,43224
Recovered,2439


In [9]:
# ============================================================
# STEP 10 — PATIENTS DATA PROFILING
# ============================================================

patients_raw = pd.read_csv(
    "MedCore_RCM_Data/Patients.csv"
)

print("========== PATIENTS DATA PROFILING ==========")

# 1. Dataset shape
print("\n1. Dataset Shape:")
print(f"Rows    : {patients_raw.shape[0]:,}")
print(f"Columns : {patients_raw.shape[1]}")


# 2. Data types
print("\n2. Data Types:")
print(patients_raw.dtypes)


# 3. Missing values
print("\n3. Missing Values:")

missing_values = (
    patients_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(patients_raw)
    * 100
).round(2)

display(missing_values)


# 4. Duplicate Patient IDs
print("\n4. Duplicate Patient IDs:")
print(
    patients_raw["Patient_ID"].duplicated().sum()
)


# 5. Unique values
print("\n5. Unique Values:")

for column in patients_raw.columns:
    print(f"\n{column}:")
    print(patients_raw[column].nunique())


# 6. First 10 records
print("\n6. First 10 Patient Records:")

display(
    patients_raw.head(10)
)

========== PATIENTS DATA PROFILING ==========

1. Dataset Shape:
Rows    : 15,000
Columns : 3

2. Data Types:
Patient_ID    object
Age            int64
Gender        object
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Patient_ID,0,0.0
Age,0,0.0
Gender,0,0.0



4. Duplicate Patient IDs:
0

5. Unique Values:

Patient_ID:
15000

Age:
68

Gender:
2

6. First 10 Patient Records:


,Patient_ID,Age,Gender
0,PAT00001,24,Female
1,PAT00002,70,Male
2,PAT00003,62,Female
3,PAT00004,47,Female
4,PAT00005,47,Male
5,PAT00006,76,Male
6,PAT00007,23,Male
7,PAT00008,65,Male
8,PAT00009,31,Male
9,PAT00010,24,Male


In [10]:
# ============================================================
# PATIENTS — BUSINESS RULE VALIDATION
# ============================================================

print("========== PATIENT BUSINESS RULE VALIDATION ==========")

# Rule 1: Age should be within a reasonable range
invalid_age = (
    (patients_raw["Age"] < 0) |
    (patients_raw["Age"] > 120)
).sum()

print("\nInvalid Age values:")
print(invalid_age)


# Rule 2: Gender should contain expected categories
expected_gender = {
    "Male",
    "Female"
}

unexpected_gender = (
    ~patients_raw["Gender"].isin(expected_gender)
).sum()

print("\nUnexpected Gender values:")
print(unexpected_gender)


# Rule 3: Patient ID should follow expected format
invalid_patient_id = (
    ~patients_raw["Patient_ID"].str.match(
        r"^PAT\d{5}$",
        na=False
    )
).sum()

print("\nInvalid Patient_ID format:")
print(invalid_patient_id)

========== PATIENT BUSINESS RULE VALIDATION ==========

Invalid Age values:
0

Unexpected Gender values:
0

Invalid Patient_ID format:
0


In [11]:
# ============================================================
# STEP 11 — PROVIDERS DATA PROFILING
# ============================================================

providers_raw = pd.read_csv(
    "MedCore_RCM_Data/Providers.csv"
)

print("========== PROVIDERS DATA PROFILING ==========")


# 1. Dataset shape
print("\n1. Dataset Shape:")
print(f"Rows    : {providers_raw.shape[0]:,}")
print(f"Columns : {providers_raw.shape[1]}")


# 2. Data types
print("\n2. Data Types:")
print(providers_raw.dtypes)


# 3. Missing values
print("\n3. Missing Values:")

missing_values = (
    providers_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(providers_raw)
    * 100
).round(2)

display(missing_values)


# 4. Duplicate Provider IDs
print("\n4. Duplicate Provider IDs:")
print(
    providers_raw["Provider_ID"].duplicated().sum()
)


# 5. Unique values
print("\n5. Unique Values:")

for column in providers_raw.columns:
    print(f"\n{column}:")
    print(providers_raw[column].nunique())


# 6. First 10 records
print("\n6. First 10 Provider Records:")

display(
    providers_raw.head(10)
)

========== PROVIDERS DATA PROFILING ==========

1. Dataset Shape:
Rows    : 40
Columns : 5

2. Data Types:
Provider_ID          object
Provider_Name        object
Location             object
Provider_Type        object
Specialty_Profile    object
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Provider_ID,0,0.0
Provider_Name,0,0.0
Location,0,0.0
Provider_Type,0,0.0
Specialty_Profile,0,0.0



4. Duplicate Provider IDs:
0

5. Unique Values:

Provider_ID:
40

Provider_Name:
40

Location:
6

Provider_Type:
2

Specialty_Profile:
8

6. First 10 Provider Records:


,Provider_ID,Provider_Name,Location,Provider_Type,Specialty_Profile
0,PR001,MedCore Provider 01,Chennai Central,Hospital,Cardiology|General Medicine
1,PR002,MedCore Provider 02,Chennai East,Hospital,Radiology|General Medicine
2,PR003,MedCore Provider 03,Chennai North,Hospital,Orthopedics|Emergency Medicine
3,PR004,MedCore Provider 04,Chennai South,Hospital,Oncology|General Medicine
4,PR005,MedCore Provider 05,Chennai West,Hospital,Neurology|General Medicine
5,PR006,MedCore Provider 06,Coimbatore,Hospital,Gastroenterology|General Medicine
6,PR007,MedCore Provider 07,Chennai Central,Hospital,General Medicine|Emergency Medicine
7,PR008,MedCore Provider 08,Chennai East,Hospital,Cardiology|Radiology|General Medicine
8,PR009,MedCore Provider 09,Chennai North,Hospital,Cardiology|General Medicine
9,PR010,MedCore Provider 10,Chennai South,Hospital,Radiology|General Medicine


In [12]:
# ============================================================
# PROVIDERS — BUSINESS RULE VALIDATION
# ============================================================

print("========== PROVIDER BUSINESS RULE VALIDATION ==========")


# Rule 1: Provider ID format
invalid_provider_id = (
    ~providers_raw["Provider_ID"].str.match(
        r"^PR\d{3}$",
        na=False
    )
).sum()

print("\nInvalid Provider_ID format:")
print(invalid_provider_id)


# Rule 2: Provider names should be unique
duplicate_provider_names = (
    providers_raw["Provider_Name"].duplicated().sum()
)

print("\nDuplicate Provider_Name values:")
print(duplicate_provider_names)


# Rule 3: Provider Type should contain expected categories
expected_provider_types = {
    "Hospital",
    "Clinic"
}

unexpected_provider_type = (
    ~providers_raw["Provider_Type"].isin(
        expected_provider_types
    )
).sum()

print("\nUnexpected Provider_Type values:")
print(unexpected_provider_type)


# Rule 4: Location should not be blank
blank_location = (
    providers_raw["Location"]
    .astype(str)
    .str.strip()
    .eq("")
).sum()

print("\nBlank Location values:")
print(blank_location)


# Rule 5: Specialty Profile should not be blank
blank_specialty = (
    providers_raw["Specialty_Profile"]
    .astype(str)
    .str.strip()
    .eq("")
).sum()

print("\nBlank Specialty_Profile values:")
print(blank_specialty)

========== PROVIDER BUSINESS RULE VALIDATION ==========

Invalid Provider_ID format:
0

Duplicate Provider_Name values:
0

Unexpected Provider_Type values:
16

Blank Location values:
0

Blank Specialty_Profile values:
0


In [13]:
# ============================================================
# INVESTIGATE PROVIDER TYPE VALUES
# ============================================================

print("========== PROVIDER TYPE INVESTIGATION ==========")

print("\nAll Provider_Type values:")

display(
    providers_raw["Provider_Type"]
    .value_counts(dropna=False)
    .to_frame("Provider_Count")
)


print("\nProviders with unexpected Provider_Type:")

expected_provider_types = {
    "Hospital",
    "Clinic"
}

unexpected_providers = providers_raw[
    ~providers_raw["Provider_Type"].isin(
        expected_provider_types
    )
]

display(unexpected_providers)

========== PROVIDER TYPE INVESTIGATION ==========

All Provider_Type values:


,Provider_Count
Provider_Type,
Hospital,24
Specialty Center,16



Providers with unexpected Provider_Type:


,Provider_ID,Provider_Name,Location,Provider_Type,Specialty_Profile
24,PR025,MedCore Provider 25,Chennai Central,Specialty Center,Cardiology|General Medicine
25,PR026,MedCore Provider 26,Chennai East,Specialty Center,Radiology|General Medicine
26,PR027,MedCore Provider 27,Chennai North,Specialty Center,Orthopedics|Emergency Medicine
27,PR028,MedCore Provider 28,Chennai South,Specialty Center,Oncology|General Medicine
28,PR029,MedCore Provider 29,Chennai West,Specialty Center,Neurology|General Medicine
29,PR030,MedCore Provider 30,Coimbatore,Specialty Center,Gastroenterology|General Medicine
30,PR031,MedCore Provider 31,Chennai Central,Specialty Center,General Medicine|Emergency Medicine
31,PR032,MedCore Provider 32,Chennai East,Specialty Center,Cardiology|Radiology|General Medicine
32,PR033,MedCore Provider 33,Chennai North,Specialty Center,Cardiology|General Medicine
33,PR034,MedCore Provider 34,Chennai South,Specialty Center,Radiology|General Medicine


In [14]:
# ============================================================
# STEP 12 — PAYERS DATA PROFILING
# ============================================================

payers_raw = pd.read_csv(
    "MedCore_RCM_Data/Payers.csv"
)

print("========== PAYERS DATA PROFILING ==========")


# 1. Dataset shape
print("\n1. Dataset Shape:")
print(f"Rows    : {payers_raw.shape[0]:,}")
print(f"Columns : {payers_raw.shape[1]}")


# 2. Data types
print("\n2. Data Types:")
print(payers_raw.dtypes)


# 3. Missing values
print("\n3. Missing Values:")

missing_values = (
    payers_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(payers_raw)
    * 100
).round(2)

display(missing_values)


# 4. Duplicate Payer IDs
print("\n4. Duplicate Payer IDs:")
print(
    payers_raw["Payer_ID"].duplicated().sum()
)


# 5. Unique values
print("\n5. Unique Values:")

for column in payers_raw.columns:
    print(f"\n{column}:")
    print(payers_raw[column].nunique())


# 6. First 10 records
print("\n6. First 10 Payer Records:")

display(
    payers_raw.head(10)
)

========== PAYERS DATA PROFILING ==========

1. Dataset Shape:
Rows    : 8
Columns : 3

2. Data Types:
Payer_ID      object
Payer_Name    object
Payer_Type    object
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Payer_ID,0,0.0
Payer_Name,0,0.0
Payer_Type,0,0.0



4. Duplicate Payer IDs:
0

5. Unique Values:

Payer_ID:
8

Payer_Name:
8

Payer_Type:
2

6. First 10 Payer Records:


,Payer_ID,Payer_Name,Payer_Type
0,PY001,SecureHealth Insurance,Private
1,PY002,PrimeCare Health,Private
2,PY003,MediSure Insurance,Private
3,PY004,WellLife Health,Private
4,PY005,NationalCare Health,Government
5,PY006,United Medical Assurance,Private
6,PY007,HealthFirst Insurance,Private
7,PY008,CareShield Health,Private


In [15]:
# ============================================================
# PAYERS — BUSINESS RULE VALIDATION
# ============================================================

print("========== PAYER BUSINESS RULE VALIDATION ==========")


# Rule 1: Payer ID format
invalid_payer_id = (
    ~payers_raw["Payer_ID"].str.match(
        r"^PY\d{3}$",
        na=False
    )
).sum()

print("\nInvalid Payer_ID format:")
print(invalid_payer_id)


# Rule 2: Payer names should be unique
duplicate_payer_names = (
    payers_raw["Payer_Name"].duplicated().sum()
)

print("\nDuplicate Payer_Name values:")
print(duplicate_payer_names)


# Rule 3: Payer Type should contain expected categories
expected_payer_types = {
    "Private",
    "Government"
}

unexpected_payer_type = (
    ~payers_raw["Payer_Type"].isin(
        expected_payer_types
    )
).sum()

print("\nUnexpected Payer_Type values:")
print(unexpected_payer_type)


# Rule 4: Blank payer names
blank_payer_name = (
    payers_raw["Payer_Name"]
    .astype(str)
    .str.strip()
    .eq("")
).sum()

print("\nBlank Payer_Name values:")
print(blank_payer_name)

========== PAYER BUSINESS RULE VALIDATION ==========

Invalid Payer_ID format:
0

Duplicate Payer_Name values:
0

Unexpected Payer_Type values:
0

Blank Payer_Name values:
0


In [16]:
# ============================================================
# STEP 13 — PROCEDURES DATA PROFILING
# ============================================================

procedures_raw = pd.read_csv(
    "MedCore_RCM_Data/Procedures.csv"
)

print("========== PROCEDURES DATA PROFILING ==========")


# 1. Dataset shape
print("\n1. Dataset Shape:")
print(f"Rows    : {procedures_raw.shape[0]:,}")
print(f"Columns : {procedures_raw.shape[1]}")


# 2. Data types
print("\n2. Data Types:")
print(procedures_raw.dtypes)


# 3. Missing values
print("\n3. Missing Values:")

missing_values = (
    procedures_raw.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"]
    / len(procedures_raw)
    * 100
).round(2)

display(missing_values)


# 4. Duplicate Procedure Codes
print("\n4. Duplicate Procedure Codes:")
print(
    procedures_raw["Procedure_Code"].duplicated().sum()
)


# 5. Duplicate Procedure Names
print("\n5. Duplicate Procedure Names:")
print(
    procedures_raw["Procedure_Name"].duplicated().sum()
)


# 6. Unique values
print("\n6. Unique Values:")

for column in procedures_raw.columns:
    print(f"\n{column}:")
    print(procedures_raw[column].nunique())


# 7. First 10 records
print("\n7. First 10 Procedure Records:")

display(
    procedures_raw.head(10)
)

========== PROCEDURES DATA PROFILING ==========

1. Dataset Shape:
Rows    : 24
Columns : 6

2. Data Types:
Procedure_Code            object
Procedure_Name            object
Specialty                 object
Authorization_Required      bool
Min_Claim_Amount           int64
Max_Claim_Amount           int64
dtype: object

3. Missing Values:


,Missing_Count,Missing_Percentage
Procedure_Code,0,0.0
Procedure_Name,0,0.0
Specialty,0,0.0
Authorization_Required,0,0.0
Min_Claim_Amount,0,0.0
Max_Claim_Amount,0,0.0



4. Duplicate Procedure Codes:
0

5. Duplicate Procedure Names:
0

6. Unique Values:

Procedure_Code:
24

Procedure_Name:
24

Specialty:
8

Authorization_Required:
2

Min_Claim_Amount:
14

Max_Claim_Amount:
17

7. First 10 Procedure Records:


,Procedure_Code,Procedure_Name,Specialty,Authorization_Required,Min_Claim_Amount,Max_Claim_Amount
0,PC001,ECG,Cardiology,False,800,2500
1,PC002,Echocardiogram,Cardiology,False,2500,6000
2,PC003,Cardiac Stress Test,Cardiology,True,3000,7500
3,PC004,Cardiac Catheterization,Cardiology,True,45000,120000
4,PC005,X-Ray,Radiology,False,500,2000
5,PC006,CT Scan,Radiology,True,5000,15000
6,PC007,MRI Brain,Radiology,True,8000,20000
7,PC008,MRI Spine,Radiology,True,9000,22000
8,PC009,Ultrasound,Radiology,False,1000,4000
9,PC010,Fracture Treatment,Orthopedics,False,8000,30000


In [17]:
# ============================================================
# PROCEDURES — BUSINESS RULE VALIDATION
# ============================================================

print("========== PROCEDURE BUSINESS RULE VALIDATION ==========")


# Rule 1: Procedure Code format
invalid_procedure_code = (
    ~procedures_raw["Procedure_Code"].str.match(
        r"^PC\d{3}$",
        na=False
    )
).sum()

print("\nInvalid Procedure_Code format:")
print(invalid_procedure_code)


# Rule 2: Procedure names should be unique
duplicate_procedure_names = (
    procedures_raw["Procedure_Name"].duplicated().sum()
)

print("\nDuplicate Procedure_Name values:")
print(duplicate_procedure_names)


# Rule 3: Minimum claim amount should not exceed maximum
invalid_amount_range = (
    procedures_raw["Min_Claim_Amount"]
    > procedures_raw["Max_Claim_Amount"]
).sum()

print("\nMin_Claim_Amount greater than Max_Claim_Amount:")
print(invalid_amount_range)


# Rule 4: Claim amount limits should be positive
invalid_min_amount = (
    procedures_raw["Min_Claim_Amount"] <= 0
).sum()

invalid_max_amount = (
    procedures_raw["Max_Claim_Amount"] <= 0
).sum()

print("\nNon-positive Min_Claim_Amount:")
print(invalid_min_amount)

print("\nNon-positive Max_Claim_Amount:")
print(invalid_max_amount)


# Rule 5: Specialty should not be blank
blank_specialty = (
    procedures_raw["Specialty"]
    .astype(str)
    .str.strip()
    .eq("")
).sum()

print("\nBlank Specialty values:")
print(blank_specialty)

========== PROCEDURE BUSINESS RULE VALIDATION ==========

Invalid Procedure_Code format:
0

Duplicate Procedure_Name values:
0

Min_Claim_Amount greater than Max_Claim_Amount:
0

Non-positive Min_Claim_Amount:
0

Non-positive Max_Claim_Amount:
0

Blank Specialty values:
0


In [18]:
# ============================================================
# STEP 14 — CROSS-TABLE REFERENCE VALIDATION
# ============================================================

print("========== CROSS-TABLE REFERENCE VALIDATION ==========")


# ------------------------------------------------------------
# 1. Claims → Patients
# ------------------------------------------------------------

missing_patients = (
    ~claims_raw["Patient_ID"].isin(
        patients_raw["Patient_ID"]
    )
).sum()

print("\nClaims with missing Patient_ID match:")
print(missing_patients)


# ------------------------------------------------------------
# 2. Claims → Providers
# ------------------------------------------------------------

missing_providers = (
    ~claims_raw["Provider_ID"].isin(
        providers_raw["Provider_ID"]
    )
).sum()

print("\nClaims with missing Provider_ID match:")
print(missing_providers)


# ------------------------------------------------------------
# 3. Claims → Payers
# ------------------------------------------------------------

missing_payers = (
    ~claims_raw["Payer_ID"].isin(
        payers_raw["Payer_ID"]
    )
).sum()

print("\nClaims with missing Payer_ID match:")
print(missing_payers)


# ------------------------------------------------------------
# 4. Claims → Procedures
# ------------------------------------------------------------

missing_procedures = (
    ~claims_raw["Procedure_Code"].isin(
        procedures_raw["Procedure_Code"]
    )
).sum()

print("\nClaims with missing Procedure_Code match:")
print(missing_procedures)


# ------------------------------------------------------------
# 5. Authorization → Claims
# ------------------------------------------------------------

authorization_missing_claims = (
    ~authorization_raw["Claim_ID"].isin(
        claims_raw["Claim_ID"]
    )
).sum()

print("\nAuthorization records without matching Claim_ID:")
print(authorization_missing_claims)


# ------------------------------------------------------------
# 6. Payments → Claims
# ------------------------------------------------------------

payment_missing_claims = (
    ~payments_raw["Claim_ID"].isin(
        claims_raw["Claim_ID"]
    )
).sum()

print("\nPayment records without matching Claim_ID:")
print(payment_missing_claims)

========== CROSS-TABLE REFERENCE VALIDATION ==========

Claims with missing Patient_ID match:
0

Claims with missing Provider_ID match:
0

Claims with missing Payer_ID match:
0

Claims with missing Procedure_Code match:
0

Authorization records without matching Claim_ID:
0

Payment records without matching Claim_ID:
0


In [19]:
# ============================================================
# CLAIMS ↔ PROCEDURES BUSINESS VALIDATION
# ============================================================

print("========== CLAIMS ↔ PROCEDURES VALIDATION ==========")


# ------------------------------------------------------------
# Bring procedure information into Claims
# ------------------------------------------------------------

claim_procedure_check = claims_raw.merge(
    procedures_raw[
        [
            "Procedure_Code",
            "Procedure_Name",
            "Specialty",
            "Authorization_Required",
            "Min_Claim_Amount",
            "Max_Claim_Amount"
        ]
    ],
    on="Procedure_Code",
    how="left",
    suffixes=("_Claim", "_Procedure")
)


# ------------------------------------------------------------
# Rule 1: Procedure name should match
# ------------------------------------------------------------

procedure_name_mismatch = (
    claim_procedure_check["Procedure_Name_Claim"]
    != claim_procedure_check["Procedure_Name_Procedure"]
).sum()

print("\nProcedure Name mismatches:")
print(procedure_name_mismatch)


# ------------------------------------------------------------
# Rule 2: Claim specialty vs procedure specialty
# ------------------------------------------------------------

specialty_mismatch = (
    claim_procedure_check["Specialty_Claim"]
    != claim_procedure_check["Specialty_Procedure"]
).sum()

print("\nSpecialty mismatches:")
print(specialty_mismatch)


# ------------------------------------------------------------
# Rule 3: Authorization requirement comparison
# ------------------------------------------------------------

authorization_required_mismatch = (
    claim_procedure_check["Authorization_Required"]
    != claim_procedure_check["Authorization_Required"]
).sum()

print("\nAuthorization requirement mismatches:")
print(authorization_required_mismatch)


# ------------------------------------------------------------
# Rule 4: Claim amount below procedure minimum
# ------------------------------------------------------------

below_minimum = (
    claim_procedure_check["Claim_Amount"]
    < claim_procedure_check["Min_Claim_Amount"]
).sum()

print("\nClaims below procedure minimum amount:")
print(below_minimum)


# ------------------------------------------------------------
# Rule 5: Claim amount above procedure maximum
# ------------------------------------------------------------

above_maximum = (
    claim_procedure_check["Claim_Amount"]
    > claim_procedure_check["Max_Claim_Amount"]
).sum()

print("\nClaims above procedure maximum amount:")
print(above_maximum)

========== CLAIMS ↔ PROCEDURES VALIDATION ==========

Procedure Name mismatches:
0

Specialty mismatches:
0

Authorization requirement mismatches:
0

Claims below procedure minimum amount:
0

Claims above procedure maximum amount:
0


In [20]:
# ============================================================
# CLAIMS ↔ PROVIDERS BUSINESS VALIDATION
# ============================================================

print("========== CLAIMS ↔ PROVIDERS VALIDATION ==========")


# ------------------------------------------------------------
# Bring provider information into Claims
# ------------------------------------------------------------

claim_provider_check = claims_raw.merge(
    providers_raw[
        [
            "Provider_ID",
            "Provider_Name",
            "Location",
            "Provider_Type",
            "Specialty_Profile"
        ]
    ],
    on="Provider_ID",
    how="left"
)


# ------------------------------------------------------------
# Check whether claim specialty is supported
# by the provider's specialty profile
# ------------------------------------------------------------

claim_provider_check["Specialty_Match"] = (
    claim_provider_check.apply(
        lambda row:
        row["Specialty"]
        in str(row["Specialty_Profile"]).split("|"),
        axis=1
    )
)


specialty_mismatch = (
    ~claim_provider_check["Specialty_Match"]
).sum()

print("\nClaims where Provider does not support Claim Specialty:")
print(specialty_mismatch)


# ------------------------------------------------------------
# Check provider names are populated after the join
# ------------------------------------------------------------

missing_provider_details = (
    claim_provider_check["Provider_Name"].isna()
).sum()

print("\nClaims with missing Provider details:")
print(missing_provider_details)


# ------------------------------------------------------------
# Show mismatches if any
# ------------------------------------------------------------

if specialty_mismatch > 0:

    print("\nProvider Specialty Mismatches:")

    display(
        claim_provider_check[
            ~claim_provider_check["Specialty_Match"]
        ][
            [
                "Claim_ID",
                "Provider_ID",
                "Provider_Name",
                "Specialty",
                "Specialty_Profile"
            ]
        ].head(20)
    )

========== CLAIMS ↔ PROVIDERS VALIDATION ==========

Claims where Provider does not support Claim Specialty:
0

Claims with missing Provider details:
0


In [22]:
# ============================================================
# CLAIMS ↔ AUTHORIZATION BUSINESS VALIDATION — CORRECTED
# ============================================================

print("========== CLAIMS ↔ AUTHORIZATION VALIDATION ==========")


# ------------------------------------------------------------
# 1. Bring Authorization information into Claims
# ------------------------------------------------------------

claim_auth_check = claims_raw.merge(
    authorization_raw[
        [
            "Claim_ID",
            "Authorization_Required",
            "Authorization_Status",
            "Request_Date",
            "Approval_Date",
            "Authorization_ID"
        ]
    ],
    on="Claim_ID",
    how="left"
)


# ------------------------------------------------------------
# 2. Bring Procedure authorization requirement
# ------------------------------------------------------------

claim_auth_check = claim_auth_check.merge(
    procedures_raw[
        [
            "Procedure_Code",
            "Authorization_Required"
        ]
    ],
    on="Procedure_Code",
    how="left",
    suffixes=(
        "_Auth",
        "_Procedure"
    )
)


# ------------------------------------------------------------
# Rule 1:
# Authorization requirement should match between
# Authorization table and Procedure table
# ------------------------------------------------------------

requirement_mismatch = (
    claim_auth_check["Authorization_Required_Auth"]
    != claim_auth_check["Authorization_Required_Procedure"]
).sum()

print("\nAuthorization requirement mismatches:")
print(requirement_mismatch)


# ------------------------------------------------------------
# Rule 2:
# Procedures requiring authorization should not be
# marked as Not Required
# ------------------------------------------------------------

required_but_not_required = (
    (claim_auth_check["Authorization_Required_Procedure"] == True) &
    (claim_auth_check["Authorization_Status"] == "Not Required")
).sum()

print("\nRequired authorization but status is Not Required:")
print(required_but_not_required)


# ------------------------------------------------------------
# Rule 3:
# Procedures not requiring authorization should have
# Not Required status
# ------------------------------------------------------------

not_required_with_status = (
    (claim_auth_check["Authorization_Required_Procedure"] == False) &
    (claim_auth_check["Authorization_Status"] != "Not Required")
).sum()

print("\nNot-required procedure with unexpected authorization status:")
print(not_required_with_status)


# ------------------------------------------------------------
# Rule 4:
# Approved authorization should have an Authorization ID
# ------------------------------------------------------------

approved_without_id = (
    (claim_auth_check["Authorization_Status"] == "Approved") &
    (claim_auth_check["Authorization_ID"].isna())
).sum()

print("\nApproved authorization without Authorization ID:")
print(approved_without_id)


# ------------------------------------------------------------
# Rule 5:
# Denied authorization should have an Authorization ID
# ------------------------------------------------------------

denied_without_id = (
    (claim_auth_check["Authorization_Status"] == "Denied") &
    (claim_auth_check["Authorization_ID"].isna())
).sum()

print("\nDenied authorization without Authorization ID:")
print(denied_without_id)


# ------------------------------------------------------------
# Rule 6:
# Authorization status distribution for procedures
# requiring authorization
# ------------------------------------------------------------

print(
    "\nAuthorization Status for Procedures "
    "Requiring Authorization:"
)

display(
    claim_auth_check[
        claim_auth_check["Authorization_Required_Procedure"] == True
    ]["Authorization_Status"]
    .value_counts(dropna=False)
    .to_frame("Claim_Count")
)

========== CLAIMS ↔ AUTHORIZATION VALIDATION ==========

Authorization requirement mismatches:
0

Required authorization but status is Not Required:
0

Not-required procedure with unexpected authorization status:
0

Approved authorization without Authorization ID:
0

Denied authorization without Authorization ID:
0

Authorization Status for Procedures Requiring Authorization:


,Claim_Count
Authorization_Status,
Approved,12663
Not Requested,1985
Pending,1164
Denied,864


In [23]:
# ============================================================
# CLAIMS ↔ PAYMENTS BUSINESS VALIDATION
# ============================================================

print("========== CLAIMS ↔ PAYMENTS VALIDATION ==========")

# Bring payment information into Claims
claim_payment_check = claims_raw.merge(
    payments_raw[
        [
            "Claim_ID",
            "Paid_Amount",
            "Outstanding_Amount",
            "Payment_Date",
            "Recovery_Status"
        ]
    ],
    on="Claim_ID",
    how="left"
)

# Rule 1: Payment record should match Claim Amount
amount_mismatch = (
    (claim_payment_check["Paid_Amount"] +
     claim_payment_check["Outstanding_Amount"])
    != claim_payment_check["Claim_Amount"]
).sum()

print("\nPayment amount reconciliation mismatches:")
print(amount_mismatch)

# Rule 2: No negative payment amounts
negative_paid = (
    claim_payment_check["Paid_Amount"] < 0
).sum()

print("\nNegative Paid Amount:")
print(negative_paid)

# Rule 3: No negative outstanding amounts
negative_outstanding = (
    claim_payment_check["Outstanding_Amount"] < 0
).sum()

print("\nNegative Outstanding Amount:")
print(negative_outstanding)

# Rule 4: Zero paid amount should not have a payment date
zero_paid_with_date = (
    (claim_payment_check["Paid_Amount"] == 0) &
    (claim_payment_check["Payment_Date"].notna())
).sum()

print("\nZero Paid Amount with Payment Date:")
print(zero_paid_with_date)

# Rule 5: Positive paid amount should have a payment date
positive_paid_without_date = (
    (claim_payment_check["Paid_Amount"] > 0) &
    (claim_payment_check["Payment_Date"].isna())
).sum()

print("\nPositive Paid Amount without Payment Date:")
print(positive_paid_without_date)

# Rule 6: Payment should not occur before Service Date
payment_before_service = (
    claim_payment_check["Payment_Date"].notna() &
    (
        claim_payment_check["Payment_Date"]
        < claim_payment_check["Service_Date"]
    )
).sum()

print("\nPayment before Service Date:")
print(payment_before_service)

# Recovery status distribution
print("\nRecovery Status:")
display(
    claim_payment_check["Recovery_Status"]
    .value_counts(dropna=False)
    .to_frame("Claim_Count")
)

========== CLAIMS ↔ PAYMENTS VALIDATION ==========

Payment amount reconciliation mismatches:
10895

Negative Paid Amount:
0

Negative Outstanding Amount:
0

Zero Paid Amount with Payment Date:
0

Positive Paid Amount without Payment Date:
0

Payment before Service Date:
0

Recovery Status:


,Claim_Count
Recovery_Status,
Collected,43224
Pending,2734
Recovered,2439
Unrecovered,1603


In [24]:
# ============================================================
# INVESTIGATE PAYMENT AMOUNT MISMATCHES
# ============================================================

payment_amount_check = claim_payment_check[
    (
        claim_payment_check["Paid_Amount"] +
        claim_payment_check["Outstanding_Amount"]
    )
    != claim_payment_check["Claim_Amount"]
].copy()

print("Number of mismatched claims:")
print(len(payment_amount_check))

print("\nSample mismatched records:")
display(
    payment_amount_check[
        [
            "Claim_ID",
            "Claim_Amount",
            "Paid_Amount",
            "Outstanding_Amount",
            "Recovery_Status"
        ]
    ].head(20)
)

Number of mismatched claims:
10895

Sample mismatched records:


,Claim_ID,Claim_Amount,Paid_Amount,Outstanding_Amount,Recovery_Status
4,CL000005,1653.12,1569.67,83.45,Collected
5,CL000006,4468.31,4200.40,267.91,Collected
11,CL000012,26741.85,26389.22,352.63,Collected
13,CL000014,15918.61,15668.89,249.72,Collected
15,CL000016,1439.89,1372.10,67.79,Collected
16,CL000017,1264.35,1239.45,24.90,Collected
17,CL000018,5071.43,5048.53,22.90,Collected
19,CL000020,3381.97,3327.86,54.11,Collected
20,CL000021,10428.26,9672.88,755.38,Collected
21,CL000022,5395.97,4783.40,612.57,Collected


In [25]:
# ============================================================
# CLAIMS ↔ PAYMENTS — CORRECTED AMOUNT VALIDATION
# ============================================================

claim_payment_check["Calculated_Claim_Amount"] = (
    claim_payment_check["Paid_Amount"] +
    claim_payment_check["Outstanding_Amount"]
)

# Compare after rounding to 2 decimal places
amount_mismatch_corrected = (
    claim_payment_check["Calculated_Claim_Amount"].round(2)
    != claim_payment_check["Claim_Amount"].round(2)
).sum()

print("Payment amount reconciliation mismatches after rounding:")
print(amount_mismatch_corrected)

Payment amount reconciliation mismatches after rounding:
0


In [26]:
# ============================================================
# CREATE CLEAN DATA FOLDER
# ============================================================

import os

clean_folder = "MedCore_RCM_Clean"

os.makedirs(clean_folder, exist_ok=True)

print("Clean data folder ready:")
print(clean_folder)

Clean data folder ready:
MedCore_RCM_Clean


In [27]:
# ============================================================
# CREATE CLEAN COPIES
# ============================================================

claims_clean = claims_raw.copy()
authorization_clean = authorization_raw.copy()
payments_clean = payments_raw.copy()
patients_clean = patients_raw.copy()
providers_clean = providers_raw.copy()
payers_clean = payers_raw.copy()
procedures_clean = procedures_raw.copy()

print("Clean copies created successfully.")

Clean copies created successfully.


In [28]:
# ============================================================
# STANDARDIZE DATE COLUMNS
# ============================================================

claims_clean["Service_Date"] = pd.to_datetime(
    claims_clean["Service_Date"],
    errors="coerce"
)

authorization_clean["Request_Date"] = pd.to_datetime(
    authorization_clean["Request_Date"],
    errors="coerce"
)

authorization_clean["Approval_Date"] = pd.to_datetime(
    authorization_clean["Approval_Date"],
    errors="coerce"
)

payments_clean["Payment_Date"] = pd.to_datetime(
    payments_clean["Payment_Date"],
    errors="coerce"
)

print("Date columns standardized.")

Date columns standardized.


In [29]:
# ============================================================
# STANDARDIZE MONETARY FIELDS
# ============================================================

claims_clean["Claim_Amount"] = claims_clean["Claim_Amount"].round(2)

payments_clean["Paid_Amount"] = payments_clean["Paid_Amount"].round(2)

payments_clean["Outstanding_Amount"] = (
    payments_clean["Outstanding_Amount"].round(2)
)

procedures_clean["Min_Claim_Amount"] = (
    procedures_clean["Min_Claim_Amount"].round(2)
)

procedures_clean["Max_Claim_Amount"] = (
    procedures_clean["Max_Claim_Amount"].round(2)
)

print("Monetary fields standardized to 2 decimal places.")

Monetary fields standardized to 2 decimal places.


In [30]:
# ============================================================
# POST-CLEANING VALIDATION — BASIC CHECKS
# ============================================================

print("========== POST-CLEANING VALIDATION ==========")

print("\nClaims shape:")
print(claims_clean.shape)

print("\nAuthorization shape:")
print(authorization_clean.shape)

print("\nPayments shape:")
print(payments_clean.shape)

print("\nPatients shape:")
print(patients_clean.shape)

print("\nProviders shape:")
print(providers_clean.shape)

print("\nPayers shape:")
print(payers_clean.shape)

print("\nProcedures shape:")
print(procedures_clean.shape)

========== POST-CLEANING VALIDATION ==========

Claims shape:
(50000, 12)

Authorization shape:
(50000, 6)

Payments shape:
(50000, 5)

Patients shape:
(15000, 3)

Providers shape:
(40, 5)

Payers shape:
(8, 3)

Procedures shape:
(24, 6)


In [31]:
# ============================================================
# CHECK MISSING VALUES AFTER CLEANING
# ============================================================

print("========== MISSING VALUE CHECK ==========")

print("\nClaims:")
display(claims_clean.isna().sum().to_frame("Missing_Count"))

print("\nAuthorization:")
display(authorization_clean.isna().sum().to_frame("Missing_Count"))

print("\nPayments:")
display(payments_clean.isna().sum().to_frame("Missing_Count"))

========== MISSING VALUE CHECK ==========

Claims:


,Missing_Count
Claim_ID,0
Patient_ID,0
Provider_ID,0
Payer_ID,0
Service_Date,0
Specialty,0
Procedure_Code,0
Procedure_Name,0
Claim_Amount,0
Denial_Flag,0



Authorization:


,Missing_Count
Claim_ID,0
Authorization_Required,0
Authorization_Status,0
Request_Date,35309
Approval_Date,37337
Authorization_ID,35309



Payments:


,Missing_Count
Claim_ID,0
Paid_Amount,0
Payment_Date,4337
Recovery_Status,0
Outstanding_Amount,0


In [32]:
# ============================================================
# FINAL PAYMENT RECONCILIATION CHECK
# ============================================================

payment_reconciliation = (
    (
        payments_clean["Paid_Amount"] +
        payments_clean["Outstanding_Amount"]
    ).round(2)
    !=
    claims_clean["Claim_Amount"].round(2)
)

print("Payment reconciliation mismatches:")
print(payment_reconciliation.sum())

Payment reconciliation mismatches:
0


In [33]:
# ============================================================
# EXPORT CLEAN DATASETS
# ============================================================

claims_clean.to_csv(
    f"{clean_folder}/Claims_Clean.csv",
    index=False
)

authorization_clean.to_csv(
    f"{clean_folder}/Authorization_Clean.csv",
    index=False
)

payments_clean.to_csv(
    f"{clean_folder}/Payments_Clean.csv",
    index=False
)

patients_clean.to_csv(
    f"{clean_folder}/Patients_Clean.csv",
    index=False
)

providers_clean.to_csv(
    f"{clean_folder}/Providers_Clean.csv",
    index=False
)

payers_clean.to_csv(
    f"{clean_folder}/Payers_Clean.csv",
    index=False
)

procedures_clean.to_csv(
    f"{clean_folder}/Procedures_Clean.csv",
    index=False
)

print("All clean datasets exported successfully.")

All clean datasets exported successfully.


In [34]:
# ============================================================
# VERIFY EXPORTED FILES
# ============================================================

print("Files in clean-data folder:")

for file in sorted(os.listdir(clean_folder)):
    print(file)

Files in clean-data folder:
Authorization_Clean.csv
Claims_Clean.csv
Patients_Clean.csv
Payers_Clean.csv
Payments_Clean.csv
Procedures_Clean.csv
Providers_Clean.csv
